# Leakage-Aware Evaluation of ML Models for Voice-Based Parkinson's Screening

Reproduces every result in the manuscript. Both datasets download automatically;
nothing needs to be uploaded.

**Runtime:** about 25 minutes on a standard Colab CPU instance. No GPU needed.

Run the cells in order. Section 6 measures timings **on this machine** and prints
a LaTeX block ready to paste into the paper.

## 1. Environment setup

In [ ]:
!pip install -q catboost xgboost lightgbm shap
!apt-get -qq install -y unrar-free > /dev/null
print("dependencies installed")

In [ ]:
# Pin threads before heavy imports so timings are hardware-comparable.
import os
for v in ("OMP_NUM_THREADS","MKL_NUM_THREADS","OPENBLAS_NUM_THREADS",
          "NUMEXPR_NUM_THREADS","VECLIB_MAXIMUM_THREADS"):
    os.environ[v] = "1"

# Option A: clone the repository (replace with your URL once published)
# !git clone https://github.com/<user>/<repo>.git && %cd <repo>

# Option B: if you uploaded the pdpipe/ folder to this session, just continue.
import sys; sys.path.insert(0, ".")
print("ready")

## 2. Load both cohorts

The Oxford cohort is the training benchmark; the Istanbul cohort is used only
for external validation. Note the composition: 32 subject identifiers, of which
just 8 are controls. That imbalance drives much of what follows.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
from pdpipe.data import load_oxford, load_istanbul, harmonised_oxford, load_oxford_meta

ox, ist = load_oxford(), load_istanbul()
print("Oxford  :", ox.summary())
print("Istanbul:", ist.summary())

### Subject parsing and the 31-vs-32 discrepancy

Every filename matches `phon_<session>_<subject>_<index>`. The documentation
reports 31 participants; the filenames yield 32 codes. The 8 control codes match
the documentation exactly, so the discrepancy sits entirely among patients.

In [ ]:
meta = load_oxford_meta()
g = meta.groupby("subject").agg(n=("filename","size"),
                                status=("status","first")).reset_index()
print(f"subject codes: {len(g)}  |  PD: {(g.status==1).sum()}  |  control: {(g.status==0).sum()}")
print(f"sessions: {meta.session.unique()}")
g.head(10)

## 3. Corrected internal benchmark

Repeated stratified group nested cross-validation: 10 repeats x 5 outer x 3 inner.
The subject is the grouping unit in **both** loops, and predictions are averaged
per participant before any metric is computed.

Set `N_REPEATS = 3` for a faster check; the published figures use 10.

In [ ]:
from pdpipe.models import build_models
from pdpipe.cv import run_nested_cv
from pdpipe.stats import summarise_models

N_REPEATS = 10   # 3 for a quick check
SEED = 20260917

res22 = run_nested_cv(ox, build_models(), n_repeats=N_REPEATS, base_seed=SEED)
summary = summarise_models(res22.fold_scores)
print(summary.round(3).to_string(index=False))

### Fold stability

This is why repetition matters: many outer test folds contain a single control
subject, and individual fold AUCs span the full range.

In [ ]:
fc = res22.fold_composition
ot = fc[fc.split == "outer_test"]
print("control subjects per outer test fold:")
print(ot.n_hc_subjects.value_counts().sort_index().to_string())
fs = res22.fold_scores
print(f"\nfold AUC range: {fs.auc_subject.min():.2f} to {fs.auc_subject.max():.2f}")
print(f"model-fold estimates at or below 0.5: {(fs.auc_subject<=0.5).sum()} of {len(fs)}")

## 4. Statistical comparison

A Wilcoxon test over five folds has almost no power. With 50 folds, plus effect
sizes, repeat-level bootstrap intervals and Holm correction across all 36
pairwise comparisons, the picture changes.

In [ ]:
from pdpipe.stats import pairwise_comparisons

pc = pairwise_comparisons(res22.fold_scores)
lr = pc[(pc.model_a=="LogisticRegression") | (pc.model_b=="LogisticRegression")]
print(lr[["model_a","model_b","mean_diff","ci_lo","ci_hi",
          "rank_biserial","p_holm"]].round(4).to_string(index=False))

## 5. External validation

Three threshold policies, reported separately. Policy A never sees an external
label and is the only genuinely blind result.

In [ ]:
from pdpipe.external import run_external, distribution_shift

ox16 = harmonised_oxford(ox)
res16 = run_nested_cv(ox16, build_models(), n_repeats=N_REPEATS, base_seed=SEED)

sel = {k:v for k,v in build_models().items()
       if k in ("LogisticRegression","RandomForest","CatBoost")}
ext = run_external(ox16, ist, sel, res16.oof_subject)
cols = ["model","policy","auc","balanced_accuracy","sensitivity","specificity","mcc"]
print(ext["table"][cols].round(3).to_string(index=False))

### Why transfer fails

Restricted to control participants, so disease status cannot explain the gap.
A feature differing by several standard deviations between cohorts among healthy
people reflects the extraction pipeline, not the illness.

In [ ]:
ds = distribution_shift(ox16, ist)
print(ds[["feature","istanbul_column","oxford_mean","istanbul_mean",
          "smd_controls"]].head(8).round(4).to_string(index=False))
print(f"\nfeatures with |SMD| > 1 among controls: {(ds.abs_smd_controls>1).sum()} of 16")

## 6. Computational cost — measured on THIS machine

Run this on the machine you will name in the paper. It prints an environment
report and writes a LaTeX block to `results/timing_table.tex`.

In [ ]:
import json, os
os.makedirs("results", exist_ok=True)
from pdpipe.timing import benchmark, environment_report

table, env = benchmark(ox, build_models(), n_repetitions=5)
print("=== Environment ===")
for k, v in env.items():
    print(f"  {k}: {v}")
print("\n=== Timings ===")
print(table[["model","end_to_end_tuning_s","fit_mean_s",
             "inference_mean_ms","peak_rss_mb"]].to_string(index=False))

table.to_csv("results/timing.csv", index=False)
with open("results/environment.json","w") as f:
    json.dump(env, f, indent=2)

## 7. Out-of-fold explainability

Explanations are recomputed on every held-out fold, using the hyperparameters
that fold selected, and aggregated with stability measures rather than read off
a single split.

In [ ]:
from pdpipe.explain import run_explanations, stability_table

e = run_explanations(ox, build_models(), res22.best_params,
                     model_names=["CatBoost","RandomForest"],
                     n_repeats=min(5, N_REPEATS))
for m in ("CatBoost","RandomForest"):
    print(f"\n=== {m}: mean |SHAP| across folds ===")
    print(stability_table(e, m, "mean_abs_shap").head(6).round(4).to_string())

## 8. Save everything

In [ ]:
for name, df in [("main22_fold_scores", res22.fold_scores),
                 ("main22_fold_composition", res22.fold_composition),
                 ("main22_best_params", res22.best_params),
                 ("main22_pairwise", pc),
                 ("main16_fold_scores", res16.fold_scores),
                 ("external", ext["table"]),
                 ("harmonisation_shift", ds),
                 ("explanations", e)]:
    df.to_csv(f"results/{name}.csv", index=False)
print("written to results/")

# On Colab, download everything as one archive:
# !zip -qr results.zip results && from google.colab import files; files.download("results.zip")